# Notebook 05 — development EMA linkage and matched HR/activity baselines

Exploratory prediction from recorded pre-EMA data, **not validated real-time prediction**. All 32 final-test participants remain sealed. Read the frozen [specification](../reports/ema_baseline_specification.md) and [plain-language report](../reports/ema_baseline/d933d3999d949477/report.md).

This notebook verifies current immutable bundles and displays aggregates and synthetic examples only. No participant rows, identifiers or current survey responses are displayed. The separate guarded entrypoint `python -m garmin_ema_baseline.pipeline` reproduces the authorized calculation using the frozen configuration; it does not tune models or overwrite existing artifacts.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from garmin_preprocessing.common import verify_bundle
from garmin_ema_baseline.linkage import score_items

BUNDLE = ROOT / "private/ema_baseline/runs/640600af77404ff6"
REPORT = ROOT / "reports/ema_baseline/d933d3999d949477"
manifest = verify_bundle(BUNDLE)
verify_bundle(REPORT)
assert manifest["linked_rows"] == 3706
print("Verified development-only linked rows:", manifest["linked_rows"])
print("No final-test response or sensor data accessed.")

## Outcome coding and completeness

Questionnaires confirm stored codes 1–5 (display labels 0–4). Raw export fields are `PSS_NERVOUS`, `PSS_COPE`, `PSS_CONTROL`; Morning/Endday questionnaire names use suffixes `_1`/`_4`. All three valid integer codes are required for nervous + (6−cope) + (6−control). The score is the study's three-item perceived-stress composite, not a clinical cutoff. Blank values remain missing; no binary target or imputation.

In [ ]:
outcomes = json.loads((REPORT / "outcomes.json").read_text())
display(pd.DataFrame(outcomes["summary"]))
display(Image(filename=str(REPORT / "coverage_and_scores.png")))

## Clearly synthetic example: prior measurements → EMA score → held-out prediction

A second-resolution actual start at 12:00 uses [11:55,12:00); a minute-resolution start uses cutoff 11:59. The synthetic readings below are wholly before the second-resolution cutoff. An observed supported counter pair 100→110 adds 10 steps. A held-out participant supplies no rows to training, scaling or category encoding. The illustrative prediction below is hypothetical, not a real fitted value.

In [ ]:
synthetic_ibi_free_hr = np.array([60., 90., 120.])
synthetic_elapsed_minutes = np.array([0., 1., 2.])
synthetic_items = pd.DataFrame({"PSS_NERVOUS": ["4", "4"], "PSS_COPE": ["4", ""], "PSS_CONTROL": ["2", "2"]})
synthetic_scores = score_items(synthetic_items)
example = pd.DataFrame([{
    "example": "synthetic only",
    "HR_mean": synthetic_ibi_free_hr.mean(),
    "HR_sample_SD": synthetic_ibi_free_hr.std(ddof=1),
    "latest_HR": synthetic_ibi_free_hr[-1],
    "slope_bpm_per_minute": np.polyfit(synthetic_elapsed_minutes, synthetic_ibi_free_hr, 1)[0],
    "supported_steps": 110 - 100,
    "composite": synthetic_scores.pss_combo.iloc[0],
    "hypothetical_prediction": 7.5,
    "absolute_error": abs(10 - 7.5),
}])
display(example)
assert synthetic_scores.pss_combo.iloc[0] == 10
assert pd.isna(synthetic_scores.pss_combo.iloc[1])
print("A missing cope answer leaves the second synthetic outcome missing.")

## Matched cohorts and fixed five-fold models

Each native block/lookback uses identical training and validation rows for its dummy/context/wearable comparisons. The secondary cohort refits every comparator on the rows with supported steps. Common cohorts hold training AND validation membership constant across lookbacks. Ridge uses alpha=1 and training-only StandardScaler/OneHotEncoder. Quality, identifiers and current answers are excluded.

MAE is primary, RMSE secondary; participant MAE weights people equally. Native lookbacks have different cohorts and cannot be directly ranked. All fold metrics and training/validation counts are retained in the aggregate JSON files.

In [ ]:
cohorts = pd.read_json(REPORT / "cohorts.json")
metrics = pd.read_json(REPORT / "metrics.json")
fold_counts = pd.read_json(REPORT / "fold_counts.json")
display(cohorts.loc[cohorts.scope.eq("native")])
display(fold_counts.loc[fold_counts.scope.eq("common") & fold_counts.lookback_minutes.eq(5)])
display(metrics.loc[metrics.fold.eq("pooled") & metrics.scope.eq("common"),
                    ["block", "lookback_minutes", "model", "mae", "rmse", "participant_mae"]])
display(Image(filename=str(REPORT / "matched_performance.png")))

## Paired uncertainty and separate recording diagnostics

Two thousand paired participant-cluster resamples give percentile 95% intervals for differences in fixed out-of-fold errors. Negative favors the added inputs. These intervals omit model-refitting uncertainty and are not external validation or multiplicity-adjusted significance. All repeated EMA predictions for a sampled participant stay together.

Two HR observations do not establish representative coverage. Even the slope minimum of three readings and a 60-second span is only numerical support. Large spans can contain long gaps; supported steps are not complete-window activity. No thresholds were changed after observing outcomes.

In [ ]:
intervals = pd.read_json(REPORT / "paired_intervals.json")
display(intervals.loc[intervals.scope.eq("native") & intervals.block.eq("hr")
                      & intervals.model.eq("context_hr") & intervals.baseline.eq("context")])
display(Image(filename=str(REPORT / "paired_differences.png")))
display(Image(filename=str(REPORT / "recording_support.png")))
print("Linked dataset:", BUNDLE.relative_to(ROOT) / "linked_development.parquet")
print("OOF predictions:", BUNDLE.relative_to(ROOT) / "out_of_fold_predictions.parquet")
print("Aggregate tables and report:", REPORT.relative_to(ROOT))

## Interpretation and next experiment

The matching median dummy has lower pooled MAE than every Ridge model in every native/common comparison. Adding HR to context gives tiny, inconsistent changes; steps add no pooled MAE improvement. This does not prove absence of physiology-related signal. Complete-case selection, provisional clocks/counters and differing survey reference periods limit interpretation.

After provider clarification, preregister a development-only measurement-support sensitivity experiment using externally justified density/gap conditions and the same matched fixed models. Do not select a lookback or open final-test data from these results. No expanded HRV, binary outcomes, tuning, imputation, feature selection, commit or push occurred.